# Normalized Data EDA — MuSe-Physio Baseline

EDA on the `.npz` files in `Processed_dataset/muse_physio_baseline/`.

| Array | Shape | Meaning |
|---|---|---|
| `x` | (samples, T, 3) | BPM, ECG, RESP — Z-score normalised |
| `y` | (samples, T, 1) | Arousal label (continuous) |
| `padding_mask` | (samples, T) | True = padded, ignore |
| `target_mask` | (samples, T, 1) | True = real label available |
| `timestamps` | (samples, T) | original recording timestamp |
| `participant_id` | (samples,) | subject identifier |
| `sample_id` | (samples,) | window index within participant |

## 1 · Setup & Load

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.05)
MODALITIES = ["BPM", "ECG", "RESP"]

DATA_DIR = os.path.join("..", "Processed_dataset", "muse_physio_baseline")

def load_split(split):
    npz = np.load(os.path.join(DATA_DIR, f"{split}.npz"), allow_pickle=False)
    return {k: npz[k] for k in npz.files}

train = load_split("train")
devel = load_split("devel")
test  = load_split("test")

norm_stats = pd.read_csv(os.path.join(DATA_DIR, "feature_normalization_stats.csv"))
with open(os.path.join(DATA_DIR, "manifest.json")) as f:
    manifest = json.load(f)

for name, d in [("train", train), ("devel", devel), ("test", test)]:
    print(f"{name:5s} | x={d['x'].shape}  y={d['y'].shape}  "
          f"padding_mask={d['padding_mask'].shape}  participants={len(set(d['participant_id']))}")

## 2 · Manifest & Normalisation Stats

In [ ]:
print("Windowing:", manifest["windowing"])
print("Normalization:", manifest["normalization"])
print()
for split, summary in manifest["splits"].items():
    print(f"{split:6s} participants={summary['participants']:>3}  samples={summary['samples']:>4}  "
          f"x_shape={summary['x_shape']}  valid_target_steps={summary['valid_target_steps']}")
print()
norm_stats

## 3 · Unmasking & Deduplication

In [ ]:
def flatten_valid(data, use_target_mask=False):
    """Flattens (samples, T, ...) arrays into a long DataFrame, one row per
    non-padded timestep. If use_target_mask=True, also restrict to timesteps
    with a real label and include the label column.
    """
    x = data["x"]
    keep = ~data["padding_mask"]
    if use_target_mask:
        keep = keep & data["target_mask"][..., 0]

    sample_id_grid    = np.repeat(data["sample_id"][:, None], x.shape[1], axis=1)
    participant_grid  = np.repeat(data["participant_id"][:, None], x.shape[1], axis=1)

    df = pd.DataFrame(x[keep], columns=MODALITIES)
    df.insert(0, "participant_id", participant_grid[keep])
    df.insert(1, "sample_id", sample_id_grid[keep])
    df.insert(2, "timestamp", data["timestamps"][keep])
    df.insert(3, "segment_id", data["segment_ids"][keep])
    if use_target_mask:
        df["label"] = data["y"][keep][:, 0]
    return df


def deduplicate(df, split_name):
    """Sliding windows overlap, so the same raw timestep can appear in several
    windows. Keep one row per (participant_id, timestamp)."""
    before = len(df)
    deduped = df.drop_duplicates(subset=["participant_id", "timestamp"]).reset_index(drop=True)
    print(f"{split_name:5s} | {before:,} rows -> {len(deduped):,} unique timesteps "
          f"({before - len(deduped):,} overlap duplicates removed)")
    return deduped


train_df = deduplicate(flatten_valid(train, use_target_mask=True), "train")
devel_df = deduplicate(flatten_valid(devel, use_target_mask=True), "devel")
test_df  = deduplicate(flatten_valid(test,  use_target_mask=False), "test")

## 4 · Feature Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Per-Timestep Feature Distributions — Train (deduplicated)", fontsize=13, fontweight="bold")

for i, name in enumerate(MODALITIES):
    ax = axes[i]
    vals = train_df[name]
    ax.hist(vals, bins=60, color="#5DADE2", edgecolor="white", alpha=0.85)
    ax.axvline(0, color="#E74C3C", lw=1.5, ls="--", label="mu=0")
    ax.set_title(name, fontsize=11, fontweight="bold")
    ax.set_xlabel("Z-score"); ax.set_ylabel("Count" if i == 0 else "")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 5 · Arousal Label Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle("Arousal Label Distribution — Deduplicated Timesteps", fontsize=13, fontweight="bold")

for ax, (name, df) in zip(axes, [("Train", train_df), ("Devel", devel_df)]):
    y = df["label"]
    ax.hist(y, bins=60, color="#2980B9", edgecolor="white", alpha=0.85, density=True)
    ax.set_title(f"{name}  (n={len(y):,} timesteps)", fontsize=11)
    ax.set_xlabel("Arousal label"); ax.set_ylabel("Density")

plt.tight_layout()
plt.show()

print(f"Train — mean={train_df['label'].mean():.4f}  std={train_df['label'].std():.4f}  "
      f"min={train_df['label'].min():.4f}  max={train_df['label'].max():.4f}")
print(f"Devel — mean={devel_df['label'].mean():.4f}  std={devel_df['label'].std():.4f}  "
      f"min={devel_df['label'].min():.4f}  max={devel_df['label'].max():.4f}")

## 7 · Descriptive Statistics

In [ ]:
windows_per_subject = pd.Series(train["participant_id"]).value_counts().sort_index()

subj_label = (
    train_df.groupby("participant_id")["label"]
    .agg(["mean", "std"])
    .sort_values("mean")
)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

axes[0].bar(windows_per_subject.index.astype(str), windows_per_subject.values, color="#2980B9")
axes[0].set_title("Sliding Windows per Participant — Train", fontsize=11)
axes[0].set_xlabel("Participant"); axes[0].set_ylabel("Windows")
axes[0].tick_params(axis="x", rotation=60, labelsize=7)

axes[1].bar(subj_label.index.astype(str), subj_label["mean"], color="#5DADE2")
axes[1].errorbar(range(len(subj_label)), subj_label["mean"], yerr=subj_label["std"],
                  fmt="none", color="#2C3E50", lw=1, capsize=2)
axes[1].set_title("Mean Arousal Label per Participant (sorted) — Train", fontsize=11)
axes[1].set_xlabel("Participant"); axes[1].set_ylabel("Arousal label")
axes[1].tick_params(axis="x", rotation=60, labelsize=7)

plt.tight_layout()
plt.show()

In [ ]:
print("--- Data Types (train_df) ---")
print(train_df.dtypes.to_string())

print("\n--- Missing Values per Split ---")
any_missing = False
for name, df in [("Train", train_df), ("Devel", devel_df), ("Test", test_df)]:
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) > 0:
        print(f"\n{name}:")
        print(missing.to_string())
        any_missing = True
    else:
        print(f"{name:5s}: no missing values")

if not any_missing:
    print("\nAll clear — zero NaNs across all splits after masking and deduplication.")

## 8 · Missing Values & Data Types

## 9 · Correlation Heatmap

In [ ]:
corr_cols = MODALITIES + ["label"]
corr_matrix = train_df[corr_cols].corr()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Correlation Analysis — Train (deduplicated)", fontsize=13, fontweight="bold")

mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True
sns.heatmap(
    corr_matrix, ax=axes[0], annot=True, fmt=".2f", cmap="coolwarm",
    vmin=-1, vmax=1, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8},
    mask=mask
)
axes[0].set_title("Lower-Triangle Pearson Correlation", fontsize=11)

label_corr = corr_matrix["label"].drop("label").sort_values()
colors = ["#E74C3C" if v > 0 else "#2980B9" for v in label_corr]
axes[1].barh(label_corr.index, label_corr.values, color=colors)
axes[1].axvline(0, color="black", lw=0.8)
axes[1].set_title("Each Feature's Linear Correlation with Arousal Label", fontsize=11)
axes[1].set_xlabel("Pearson r")
for i, v in enumerate(label_corr.values):
    axes[1].text(v + (0.003 if v >= 0 else -0.003), i, f"{v:.3f}",
                 va="center", ha="left" if v >= 0 else "right", fontsize=9)

plt.tight_layout()
plt.show()

print("\nKey takeaway:")
strongest = label_corr.abs().idxmax()
print(f"  Strongest feature-label link: {strongest} (r={label_corr[strongest]:.3f})")
print(f"  If |r| < 0.15 across all features → linear models will struggle, non-linear needed")

## 10 · Feature vs Label Scatter

In [ ]:
from scipy import stats as sp_stats

sample = train_df.sample(n=min(5000, len(train_df)), random_state=42)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Feature vs Arousal Label — Scatter + Regression (train, n=5k sample)", fontsize=13, fontweight="bold")

for col, ax in zip(MODALITIES, axes[0]):
    x, y = sample[col].values, sample["label"].values
    slope, intercept, r, p, _ = sp_stats.linregress(x, y)
    ax.scatter(x, y, alpha=0.15, s=8, color="#5DADE2")
    xline = np.linspace(x.min(), x.max(), 200)
    ax.plot(xline, slope * xline + intercept, color="#E74C3C", lw=2, label=f"r={r:.3f}")
    ax.set_title(col, fontsize=11, fontweight="bold")
    ax.set_xlabel("Z-score"); ax.set_ylabel("Arousal label")
    ax.legend(fontsize=9)

train_df["label_bin"] = pd.qcut(train_df["label"], q=4,
                                 labels=["Q1\n(low)", "Q2", "Q3", "Q4\n(high)"])
for col, ax in zip(MODALITIES, axes[1]):
    groups = [train_df.loc[train_df["label_bin"] == b, col].values
              for b in ["Q1\n(low)", "Q2", "Q3", "Q4\n(high)"]]
    bp = ax.boxplot(groups, patch_artist=True, medianprops={"color": "black", "lw": 2},
                    flierprops={"marker": ".", "markersize": 2, "alpha": 0.3})
    colors_box = ["#AED6F1", "#5DADE2", "#2E86C1", "#1A5276"]
    for patch, c in zip(bp["boxes"], colors_box):
        patch.set_facecolor(c)
    ax.set_xticklabels(["Q1\n(low arousal)", "Q2", "Q3", "Q4\n(high arousal)"])
    ax.set_title(f"{col} by Arousal Quartile", fontsize=11, fontweight="bold")
    ax.set_ylabel("Feature Z-score")

train_df.drop(columns=["label_bin"], inplace=True)
plt.tight_layout()
plt.show()

## 11 · Train vs Devel Distribution

In [ ]:
from scipy.stats import ks_2samp

compare_cols = MODALITIES + ["label"]
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle("Train vs Devel Distribution Overlap (KDE)", fontsize=13, fontweight="bold")

ks_results = []
for ax, col in zip(axes, compare_cols):
    tr = train_df[col].dropna()
    dv = devel_df[col].dropna()
    stat, pval = ks_2samp(tr, dv)
    ks_results.append({"Feature": col, "KS stat": round(stat, 4), "p-value": pval,
                        "Verdict": "DIFFERENT" if pval < 0.05 else "similar"})
    tr.plot.kde(ax=ax, label="Train", color="#2980B9", lw=2)
    dv.plot.kde(ax=ax, label="Devel", color="#E67E22", lw=2, ls="--")
    ax.set_title(f"{col}\nKS={stat:.3f}  p={pval:.2e}", fontsize=10)
    ax.set_xlabel("Z-score" if col != "label" else "Arousal label")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print("\nKS Test Summary:")
display(pd.DataFrame(ks_results).set_index("Feature"))
print("\nIf all features show DIFFERENT: cross-subject generalisation is the core challenge.")

## 12 · Outlier Detection (IQR)

In [ ]:
outlier_rows = []
outlier_flags = pd.DataFrame(index=train_df.index)

for col in MODALITIES:
    q1, q3 = train_df[col].quantile(0.25), train_df[col].quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mask_out = (train_df[col] < lo) | (train_df[col] > hi)
    outlier_flags[col] = mask_out
    n = mask_out.sum()
    pct = 100 * n / len(train_df)
    outlier_rows.append({"Feature": col, "Q1": round(q1,3), "Q3": round(q3,3),
                          "IQR": round(iqr,3), "Lower fence": round(lo,3),
                          "Upper fence": round(hi,3), "Outliers (n)": n,
                          "Outliers (%)": round(pct,2)})

display(pd.DataFrame(outlier_rows).set_index("Feature"))

any_outlier = outlier_flags.any(axis=1)
print(f"\nTimesteps with at least one outlier feature: "
      f"{any_outlier.sum():,} / {len(train_df):,} ({100*any_outlier.mean():.2f}%)")

# Visualise: show feature distributions with fences marked
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Outlier Fences (IQR method) — Train", fontsize=13, fontweight="bold")

for ax, row in zip(axes, outlier_rows):
    col = row["Feature"]
    vals = train_df[col]
    ax.hist(vals, bins=60, color="#AED6F1", edgecolor="white", alpha=0.85)
    ax.axvline(row["Lower fence"], color="#E74C3C", lw=1.5, ls="--", label="Q1-1.5IQR")
    ax.axvline(row["Upper fence"], color="#E74C3C", lw=1.5, ls="--", label="Q3+1.5IQR")
    ax.set_title(f"{col}  ({row['Outliers (%)']:.1f}% flagged)", fontsize=11)
    ax.set_xlabel("Z-score"); ax.set_ylabel("Count" if ax == axes[0] else "")
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

## 13 · Temporal Patterns per Participant

In [ ]:
all_pids = sorted(train_df["participant_id"].unique())
pick_pids = [all_pids[0], all_pids[len(all_pids)//3],
             all_pids[2*len(all_pids)//3], all_pids[-1]]

signal_colors = {"BPM": "#2980B9", "ECG": "#27AE60", "RESP": "#8E44AD", "label": "#E74C3C"}

fig, axes = plt.subplots(4, 4, figsize=(20, 14), sharex=False)
fig.suptitle("Per-Participant Time Series — Features & Arousal Label (Train, 4 subjects)",
             fontsize=13, fontweight="bold")

for row_i, pid in enumerate(pick_pids):
    pdata = train_df[train_df["participant_id"] == pid].sort_values("timestamp")
    t_sec = pdata["timestamp"].values / 1000
    for col_i, (col, color) in enumerate(signal_colors.items()):
        ax = axes[row_i][col_i]
        ax.plot(t_sec, pdata[col].values, color=color, lw=0.8, alpha=0.85)
        if row_i == 0:
            ax.set_title("Arousal label" if col == "label" else col, fontsize=11, fontweight="bold")
        if col_i == 0:
            ax.set_ylabel(f"Subject {pid}", fontsize=9)
        if row_i == 3:
            ax.set_xlabel("Time (s)")
        ax.tick_params(labelsize=7)

plt.tight_layout()
plt.show()

print("\nInspect the plots above:")
print("  - Do features drift smoothly or jump abruptly?")
print("  - Does the arousal label tend to follow BPM with a delay?")
print("  - Do all 4 subjects show similar temporal structure, or are they very different?")

## 14 · Autocorrelation Analysis (ACF)

In [ ]:
MAX_LAG = 100
acf_cols = MODALITIES + ["label"]
acf_colors = ["#2980B9", "#27AE60", "#8E44AD", "#E74C3C"]

def acf_manual(series, max_lag):
    s = series - series.mean()
    denom = np.dot(s, s)
    return np.array([np.dot(s[:len(s)-k], s[k:]) / denom for k in range(max_lag + 1)])

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle(f"Autocorrelation Function (ACF) — First participant in Train  "
             f"(lag 0–{MAX_LAG}, = 0–{MAX_LAG/2:.0f} s at 2 Hz)",
             fontsize=12, fontweight="bold")

first_pid = all_pids[0]
pdata = train_df[train_df["participant_id"] == first_pid].sort_values("timestamp")
conf_band = 1.96 / np.sqrt(len(pdata))

for ax, col, color in zip(axes, acf_cols, acf_colors):
    acf_vals = acf_manual(pdata[col].values, MAX_LAG)
    lags = np.arange(MAX_LAG + 1)
    ax.bar(lags, acf_vals, color=color, alpha=0.7, width=1.0)
    ax.axhline(conf_band,  color="black", lw=1, ls="--", label="95% CI")
    ax.axhline(-conf_band, color="black", lw=1, ls="--")
    ax.axhline(0, color="black", lw=0.5)
    ax.set_title(col, fontsize=11, fontweight="bold")
    ax.set_xlabel("Lag (timesteps)")
    ax.set_ylabel("ACF" if ax == axes[0] else "")
    ax.legend(fontsize=7)
    ax.set_ylim(-0.3, 1.05)

plt.tight_layout()
plt.show()

# Also print the lag at which ACF first drops below the CI band
print("\nLag at which signal first drops below 95% CI threshold:")
for col, color in zip(acf_cols, acf_colors):
    acf_vals = acf_manual(pdata[col].values, MAX_LAG)
    drop_lag = next((k for k in range(1, MAX_LAG+1) if abs(acf_vals[k]) < conf_band), MAX_LAG)
    print(f"  {col:6s}: lag {drop_lag:>4}  ({drop_lag/2:.1f} s)  "
          f"— {'strong memory' if drop_lag > 20 else 'moderate memory' if drop_lag > 5 else 'weak memory'}")

## 15 · PCA — Dimensionality Reduction

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X = train_df[MODALITIES].values
y_label = train_df["label"].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)

idx = np.random.default_rng(42).choice(len(X_pca), size=min(8000, len(X_pca)), replace=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("PCA of Physiological Features — Train", fontsize=13, fontweight="bold")

sc = axes[0].scatter(X_pca[idx, 0], X_pca[idx, 1], c=y_label[idx],
                     cmap="coolwarm", alpha=0.3, s=8)
plt.colorbar(sc, ax=axes[0], label="Arousal label")
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
axes[0].set_title("PC1 vs PC2 (coloured by Arousal label)", fontsize=11)

loadings = pca.components_.T
scale = 3
for i, feat in enumerate(MODALITIES):
    axes[0].annotate("", xy=(loadings[i,0]*scale, loadings[i,1]*scale), xytext=(0,0),
                     arrowprops=dict(arrowstyle="->", color="black", lw=1.5))
    axes[0].text(loadings[i,0]*scale*1.15, loadings[i,1]*scale*1.15,
                 feat, fontsize=9, fontweight="bold")

axes[1].bar(["PC1","PC2","PC3"], pca.explained_variance_ratio_*100, color=["#2980B9","#5DADE2","#AED6F1"])
axes[1].plot(["PC1","PC2","PC3"], np.cumsum(pca.explained_variance_ratio_)*100,
             "o-", color="#E74C3C", lw=2, label="Cumulative")
axes[1].set_ylabel("Explained variance (%)")
axes[1].set_title("Scree Plot", fontsize=11)
axes[1].legend(); axes[1].set_ylim(0, 105)
for i, v in enumerate(pca.explained_variance_ratio_*100):
    axes[1].text(i, v+1.5, f"{v:.1f}%", ha="center", fontsize=9)

sc2 = axes[2].scatter(X_pca[idx, 0], X_pca[idx, 2], c=y_label[idx],
                      cmap="coolwarm", alpha=0.3, s=8)
plt.colorbar(sc2, ax=axes[2], label="Arousal label")
axes[2].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
axes[2].set_ylabel(f"PC3 ({pca.explained_variance_ratio_[2]*100:.1f}%)")
axes[2].set_title("PC1 vs PC3 (coloured by Arousal label)", fontsize=11)

plt.tight_layout()
plt.show()

print(f"\nTotal variance captured by 2 PCs: {sum(pca.explained_variance_ratio_[:2])*100:.1f}%")
print(f"Total variance captured by 3 PCs: {sum(pca.explained_variance_ratio_)*100:.1f}%")
print("\nComponent loadings (how much each feature contributes to each PC):")
display(pd.DataFrame(pca.components_.T, index=MODALITIES,
                     columns=["PC1","PC2","PC3"]).round(3))

## 16 · Mutual Information & Feature Importance

In [ ]:
from sklearn.feature_selection import mutual_info_regression
from sklearn.ensemble import RandomForestRegressor

X_tr = train_df[MODALITIES].values
y_tr = train_df["label"].values

# -- Mutual Information --
mi_scores = mutual_info_regression(X_tr, y_tr, random_state=42)
mi_df = pd.DataFrame({"Feature": MODALITIES, "Mutual Information": mi_scores}).set_index("Feature")

# -- Random Forest feature importance (fast: small forest) --
rf = RandomForestRegressor(n_estimators=100, max_depth=8, n_jobs=-1, random_state=42)
rf.fit(X_tr, y_tr)
rf_df = pd.DataFrame({"Feature": MODALITIES, "RF Importance": rf.feature_importances_}).set_index("Feature")

# -- Pearson r for comparison --
pearson_df = pd.DataFrame({"Feature": MODALITIES,
                            "Pearson |r|": [abs(np.corrcoef(X_tr[:, i], y_tr)[0,1])
                                            for i in range(3)]}).set_index("Feature")

importance_df = pearson_df.join(mi_df).join(rf_df).round(4)
print("Feature Importance Summary (Train):")
display(importance_df)

# Plot all three metrics side by side
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Feature Importance: Linear vs Non-Linear vs Model-Based", fontsize=13, fontweight="bold")
metrics = [("Pearson |r|", "#2980B9"), ("Mutual Information", "#27AE60"), ("RF Importance", "#E74C3C")]

for ax, (metric, color) in zip(axes, metrics):
    vals = importance_df[metric].sort_values(ascending=True)
    ax.barh(vals.index, vals.values, color=color, alpha=0.85)
    ax.set_title(metric, fontsize=11, fontweight="bold")
    ax.set_xlabel("Score")
    for i, v in enumerate(vals.values):
        ax.text(v + 0.001, i, f"{v:.4f}", va="center", fontsize=9)

plt.tight_layout()
plt.show()

print(f"\nRandom Forest R² on train: {rf.score(X_tr, y_tr):.4f}")
print("(A high R² here means the features DO predict the label — even if Pearson r was low)")
print("(A low R² here means arousal prediction genuinely needs sequence context, not just per-timestep values)")

## 17 · Export Deduplicated Data

In [ ]:
from IPython.display import FileLink, display

EXPORT_DIR = os.path.join(".", "exports")
os.makedirs(EXPORT_DIR, exist_ok=True)

export_paths = {}
for name, df in [("train", train_df), ("devel", devel_df), ("test", test_df)]:
    path = os.path.join(EXPORT_DIR, f"{name}_flattened.csv")
    df.to_csv(path, index=False)
    export_paths[name] = path
    print(f"Saved {name:5s} -> {path}  ({len(df):,} rows, {df.shape[1]} columns)")

print("\nDownload:")
for name, path in export_paths.items():
    display(FileLink(path))